In [1]:
import os
from model import Model

import numpy as np
import pandas as pd

SUBJECTS = ['A', 'B', 'C', 'D', 'E', 'F']

input_dir = "data"
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)


/home/felix/miniconda3/envs/islachal/lib/python3.9/site-packages/sklearn/utils/fixes.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version  # type: ignore


In [2]:
subject = 'A'  # Change this to select a different subject
X_train = np.load(os.path.join(input_dir, f"subject_{subject}_X_train.npy"))
y_train = np.load(os.path.join(input_dir, f"subject_{subject}_y_train.npy"))
model = Model()
print(X_train.shape)
print(X_train[0])

(140, 64, 1537)
[[-374.32271918 -454.97680374 -483.37062932 ... -183.06522309
  -214.86571658 -230.26149715]
 [-229.85679241 -390.00701074 -505.81294301 ... -231.03218378
  -248.67163028 -257.77524598]
 [ -78.51635985 -154.76738425 -208.42504468 ... -167.0894054
  -164.31723511 -146.26739644]
 ...
 [ -74.9770014   -79.65577432  -83.24613685 ... -104.87809995
  -153.07641083 -194.76814056]
 [ -24.04300362  -26.00096321  -29.21066106 ...  -94.38366602
  -134.60776611 -168.42624776]
 [-111.49180929 -124.43313587 -133.7321329  ... -147.44447529
  -187.78489978 -220.78777082]]


In [4]:

for subject in SUBJECTS:
    print(f'Subject {subject}: loading data')
    X_train = np.load(os.path.join(input_dir, f'subject_{subject}_X_train.npy'))
    y_train = np.load(os.path.join(input_dir, f'subject_{subject}_y_train.npy'))
    X_test = np.load(os.path.join(input_dir, f'subject_{subject}_X_test.npy'))


    m = Model()
    print(f'Subject {subject}: training')
    m.fit(X_train, y_train)
    print(f'Subject {subject}: predicting')
    y_pred = m.predict(X_test)

    out_path = os.path.join(output_dir, f'subject_{subject}_y_pred.csv')
    pd.DataFrame({'y_pred': y_pred}).to_csv(out_path, index=False)
    print(f'Subject {subject}: done ({len(y_pred)} predictions saved)')

# zip the output directory
import shutil
shutil.make_archive(output_dir, 'zip', output_dir)
print(f'Output directory "{output_dir}" zipped to "{output_dir}.zip"')

Subject A: loading data
Subject A: training
Subject A: predicting
Subject A: done (60 predictions saved)
Subject B: loading data
Subject B: training
Subject B: predicting
Subject B: done (60 predictions saved)
Subject C: loading data
Subject C: training
Subject C: predicting
Subject C: done (60 predictions saved)
Subject D: loading data
Subject D: training
Subject D: predicting
Subject D: done (60 predictions saved)
Subject E: loading data
Subject E: training
Subject E: predicting
Subject E: done (60 predictions saved)
Subject F: loading data
Subject F: training
Subject F: predicting
Subject F: done (60 predictions saved)
Output directory "output" zipped to "output.zip"


In [ ]:
from EEGNet import EEGNet
import trainEEPNet
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim

def eep_train_subject(subject, device, num_classes, batch_size):
    print(f"Training model for subject {subject}...")
    X_train = np.load(f"data/subject_{subject}_X_train.npy")
    y_train = np.load(f"data/subject_{subject}_y_train.npy")

    num_channels = X_train.shape[1]
    num_timesteps = X_train.shape[2]
    
    print(f"Data shape: {X_train.shape}, Labels shape: {y_train.shape}")
    print(f"Unique labels: {np.unique(y_train)}")

    # Initialize EEGNet model with correct dimensions
    model = EEGNet(num_temporal_filts=32, num_spatial_filts=4, num_chans=num_channels, 
                   window_length=num_timesteps, avgpool_factor=2, num_classes=num_classes)
    model.to(device)

    # Use BCIDataset with actual data (not synthetic EEGDataset)
    train_dataset = trainEEPNet.BCIDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Loss function & optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training loop
    for epoch in range(num_epochs):
        train_loss, train_acc = trainEEPNet.train(model, train_loader, criterion, optimizer, device)
        print(f"Epoch {epoch+1}/{num_epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    return model


def eep_predict(model, subject, device):
    X_test = np.load(f"data/subject_{subject}_X_test.npy")
    # Create dummy labels (not used for prediction)
    test_dataset = trainEEPNet.BCIDataset(X_test, np.zeros(len(X_test), dtype=object))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    predictions = trainEEPNet.predict(model, test_loader, device)
    return predictions



In [4]:
print(torch.cuda.is_available())

True


In [9]:


subject = 'A'  # Change this to the desired subject (A-F)
# Hyperparameters
num_epochs = 1
batch_size = 8
learning_rate = 0.001
num_classes = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    torch.cuda.empty_cache()  # Clear GPU memory before training
print(f"Using device: {device}")

model = eep_train_subject(subject, device, num_classes, batch_size)
predictions = eep_predict(model, subject, device)
print(f"Predictions on test set for subject {subject}: {predictions[:10]}")  # Print first 10 predictions

Using device: cuda
Training model for subject A...
Data shape: (140, 64, 1537), Labels shape: (140,)
Unique labels: ['left_hand' 'right_hand']
left_hand
tensor(0)
Epoch 1/1: Train Loss: 0.8734, Train Acc: 0.4857
Predictions on test set for subject A: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [10]:
# Hyperparameters
num_epochs = 10
batch_size = 8
learning_rate = 0.001
num_classes = 2

for subject in SUBJECTS:
    print(f"Processing subject {subject}...")
    model = eep_train_subject(subject, device, num_classes, batch_size)
    predictions = eep_predict(model, subject, device)
    predictions = ['left_hand' if pred == 0 else 'right_hand' for pred in predictions]
    print(f"Predictions for subject {subject}: {predictions[:10]}")
    out_path = os.path.join(output_dir, f'subject_{subject}_y_pred.csv')
    pd.DataFrame({'y_pred': predictions}).to_csv(out_path, index=False)
    print(f"Subject {subject} done, predictions saved to {out_path}")

shutil.make_archive(output_dir, 'zip', output_dir)
print(f'Output directory "{output_dir}" zipped to "{output_dir}.zip"')

Processing subject A...
Training model for subject A...
Data shape: (140, 64, 1537), Labels shape: (140,)
Unique labels: ['left_hand' 'right_hand']
left_hand
tensor(0)
Epoch 1/10: Train Loss: 0.7613, Train Acc: 0.4929
Epoch 2/10: Train Loss: 0.7021, Train Acc: 0.5214
Epoch 3/10: Train Loss: 0.7149, Train Acc: 0.5214
Epoch 4/10: Train Loss: 0.6977, Train Acc: 0.4929
Epoch 5/10: Train Loss: 0.7036, Train Acc: 0.5214
Epoch 6/10: Train Loss: 0.6873, Train Acc: 0.5429
Epoch 7/10: Train Loss: 0.6990, Train Acc: 0.5000
Epoch 8/10: Train Loss: 0.7189, Train Acc: 0.4857
Epoch 9/10: Train Loss: 0.6856, Train Acc: 0.4929
Epoch 10/10: Train Loss: 0.6791, Train Acc: 0.5429
Predictions for subject A: ['right_hand', 'right_hand', 'right_hand', 'right_hand', 'right_hand', 'left_hand', 'right_hand', 'left_hand', 'left_hand', 'right_hand']
Subject A done, predictions saved to output/subject_A_y_pred.csv
Processing subject B...
Training model for subject B...
Data shape: (140, 64, 1537), Labels shape: (1

NameError: name 'shutil' is not defined

In [11]:
import shutil
shutil.make_archive(output_dir, 'zip', output_dir)
print(f'Output directory "{output_dir}" zipped to "{output_dir}.zip"')

Output directory "output" zipped to "output.zip"
